### SQL as Data Instead of Hard‑Coded Strings

Most beginner SQL examples hard‑code everything: the table name, the field names, and the queries themselves. That works for tiny demos, but it doesn’t scale. Real programs need to build tables, insert rows, update values, and run queries based on the data structures already in the program.

The important idea is that SQL is just text. Python is good at generating text. If you treat SQL as data, you can assemble queries from the objects you already have instead of rewriting the same SQL patterns over and over.

In this section, we’ll look at how to take a Python data structure—like a namedtuple or a class—and generate SQL dynamically from it. This lets you create tables, insert rows, update fields, and run searches without scattering literal SQL strings throughout your code.

Note: The code blocks in this notebook are code excerpts taken from SqliteDB.py example file.


### Representing Rows with Namedtuples

If you want to generate SQL automatically, you need a Python structure that tells you two things:

1. **What the fields are called**
2. **What order the fields appear in**

A namedtuple is perfect for this. It behaves like a tiny record type: each field has a name, each instance has values, and the fields always appear in the same order. That makes it easy to build SQL strings from the namedtuple’s metadata instead of hard‑coding field names.

For example, if you define a namedtuple with fields `name`, `distance_au`, `orbit_years`, and `image_blob`, you can use those field names to generate a `CREATE TABLE` statement, an `INSERT` statement, or an `UPDATE` statement without writing any SQL by hand.


In [ ]:
from collections import namedtuple

PlanetTuple = namedtuple(
    "PlanetTuple",
    ["name", "distance_au", "orbit_years", "image_blob"]
)

# Example instance
earth = PlanetTuple("Earth", 1.0, 1.0, b"...binary image...")
earth


### Inferring SQLite Types from Python Values

If you want to generate a `CREATE TABLE` statement automatically, you need to know what SQLite type each field should use. Instead of hard‑coding types like `TEXT`, `REAL`, or `BLOB`, you can infer them from the Python values in your namedtuple.

This works because Python already knows the type of each attribute. If a field contains a string, it maps naturally to `TEXT`. If it contains a float, it maps to `REAL`. If it contains raw bytes—like an image file—it maps to `BLOB`.

By inspecting a sample instance of the namedtuple, you can build a small function that converts Python types into SQLite types. This lets you generate table definitions dynamically without writing any SQL by hand.


In [ ]:
def sqlite_type(value):
    pytype = type(value).__name__
    return {
        "str": "TEXT",
        "int": "INTEGER",
        "float": "REAL",
        "bytes": "BLOB"
    }.get(pytype, "TEXT")


### Building a `CREATE TABLE` Statement Dynamically

Once you have a namedtuple that defines the field names, and a type‑mapping function that converts Python values into SQLite types, you can generate a full `CREATE TABLE` statement automatically. This is the first place where treating SQL as data really pays off.

The general pattern looks like this:

1. Take the namedtuple’s field names.
2. Look at a sample instance to infer each field’s SQLite type.
3. Assemble the field definitions into a comma‑separated list.
4. Wrap the whole thing in a `CREATE TABLE` statement.

The result is a table definition that adapts to whatever fields your namedtuple contains. If you add a new attribute to the namedtuple, the SQL changes automatically. No hard‑coded SQL, no repeated field names, and no manual updates when the data model evolves.


In [ ]:
def create_table(conn, ntuple, sample_instance):
    table_name = ntuple.__name__.lower() + "s"
    fields = ntuple._fields

    column_defs = []
    for field in fields:
        value = getattr(sample_instance, field)
        column_type = sqlite_type(value)
        column_defs.append(f"{field} {column_type}")

    sql = (
        f"CREATE TABLE IF NOT EXISTS {table_name} ("
        "id INTEGER PRIMARY KEY AUTOINCREMENT, "
        + ", ".join(column_defs) +
        ");"
    )

    conn.execute(sql)
    conn.commit()
    return table_name


### Building an `INSERT` Statement Dynamically

Once you have a table created from a namedtuple, the next step is inserting rows. The pattern is the same: instead of hard‑coding field names, you assemble the SQL from the namedtuple’s metadata.

Every namedtuple instance already contains:

- the field names (`_fields`)
- the field values (the tuple itself)
- the correct ordering

That means you can generate an `INSERT` statement automatically:

1. Join the field names into a comma‑separated list.
2. Create one `?` placeholder per field.
3. Use the namedtuple instance itself as the parameter tuple.

This approach scales cleanly. If you add a new field to the namedtuple, the SQL adjusts automatically. No manual edits, no repeated strings, and no risk of mismatched field names.


In [ ]:
def insert_row(conn, table_name, ntuple_instance):
    fields = ntuple_instance._fields
    field_list = ", ".join(fields)
    placeholders = ", ".join(["?"] * len(fields))

    sql = (
        f"INSERT INTO {table_name} ({field_list}) "
        f"VALUES ({placeholders})"
    )

    conn.execute(sql, tuple(ntuple_instance))
    conn.commit()


### Building an `UPDATE` Statement Dynamically

Updating a row is slightly more complex than inserting one, because you don’t just supply values — you also need to specify which fields should change. A Python dictionary is a natural fit for this: the keys are the field names, and the values are the new data.

The general pattern for a dynamic `UPDATE` looks like this:

1. Take the dictionary of new values.
2. Convert each key–value pair into a fragment like `field = ?`.
3. Join those fragments into a comma‑separated list.
4. Add a `WHERE` clause to identify which row to update.
5. Build the parameter list by combining the new values with the lookup value.

This approach scales cleanly. If you update one field, the SQL adjusts. If you update three fields, the SQL adjusts. No hard‑coded assignments, no repeated field names, and no manual edits when the data model changes.


In [ ]:
def update_row(conn, table_name, name, new_values):
    # Build "field = ?" fragments
    assignments = ", ".join([f"{key} = ?" for key in new_values.keys()])

    # Parameter list: new values + lookup value
    params = list(new_values.values()) + [name]

    sql = (
        f"UPDATE {table_name} "
        f"SET {assignments} "
        f"WHERE name = ?"
    )

    conn.execute(sql, params)
    conn.commit()


### Building a `DELETE` Statement Dynamically

Deleting a row is the simplest SQL operation, but it still benefits from the same dynamic approach. Instead of hard‑coding a field name into the SQL, you treat the table name and the lookup value as data and assemble the query from them.

The pattern is straightforward:

1. Build a `DELETE FROM table` string using the table name.
2. Add a `WHERE` clause that uses a parameter placeholder.
3. Pass the lookup value (such as a planet’s name) as the parameter.

This keeps the SQL flexible. If your table name changes, the SQL adjusts. If your lookup field changes, the SQL adjusts. You avoid scattering literal SQL strings throughout your code, and you keep the logic consistent with the dynamic patterns used for `INSERT` and `UPDATE`.


In [ ]:
def delete_row(conn, table_name, name):
    sql = f"DELETE FROM {table_name} WHERE name = ?"
    conn.execute(sql, (name,))
    conn.commit()


### Building a `SELECT` Statement Dynamically

Selecting rows is the simplest SQL operation, but it still follows the same pattern as the others: treat the table name and any search criteria as data, and assemble the SQL from those pieces. The basic version—selecting all rows—only needs the table name. More advanced versions can add `WHERE` clauses, ordering, or limits, but the core idea stays the same.

The benefit of building `SELECT` dynamically is consistency. If your table name changes, the query adjusts automatically. If your namedtuple gains new fields, you don’t need to rewrite any SQL. You keep the logic clean and avoid scattering literal SQL strings throughout your code.


In [ ]:
def select_all(conn, table_name):
    sql = f"SELECT * FROM {table_name}"
    cur = conn.cursor()
    cur.execute(sql)
    return cur.fetchall()


### The General Pattern Behind All Dynamic SQL

At this point, every CRUD operation—create, insert, update, delete, and select—follows the same underlying pattern. Once you see the pattern, SQL stops looking like a special language and starts looking like structured text that you assemble from Python data.

The pattern is always:

1. **Start with a Python structure**
   Something that already knows its field names and values.
   A namedtuple is ideal because it gives you both.

2. **Extract the metadata you need**
   - field names
   - field values
   - Python types
   - table name
   These pieces become the building blocks of your SQL.

3. **Assemble SQL fragments from those pieces**
   - `"field = ?"`
   - `"name REAL"`
   - `"INSERT INTO table (fields) VALUES (placeholders)"`
   - `"UPDATE table SET assignments WHERE name = ?"`
   Each fragment is just text.

4. **Join the fragments into a complete SQL statement**
   Python’s string operations do the heavy lifting.
   No hard‑coded SQL, no repeated field names.

5. **Execute the statement with parameters**
   The namedtuple instance or dictionary supplies the values.
   SQLite handles the rest.

Once you understand this pattern, you can generate any SQL you need—joins, filters, ordering, limits, even schema migrations—without writing literal SQL strings all over your code. SQL becomes data, and Python becomes the tool that assembles it.


### Why Dynamic SQL?

Once you’ve seen how to generate `CREATE TABLE`, `INSERT`, `UPDATE`, `DELETE`, and `SELECT` statements from Python data, the bigger picture becomes clear: dynamic SQL isn’t a trick, it’s a way to keep your program maintainable.

Hard‑coded SQL scatters field names everywhere. If you rename a field, you have to hunt through your codebase and fix every query by hand. If you add a new attribute to your data model, you have to update every `INSERT` and `UPDATE` statement. This is brittle and error‑prone.

Dynamic SQL solves that by treating the data model as the single source of truth. The namedtuple defines the fields. A sample instance defines the types. The SQL is assembled from those pieces. If the data model changes, the SQL changes automatically. You don’t rewrite queries; you regenerate them.

This approach scales cleanly as your program grows. It also teaches an important lesson: SQL is not a separate world. It’s text. Python is good at generating text. When you combine the two, you get a flexible system that adapts to your data instead of forcing you to rewrite your queries every time your data changes.


### Putting the Dynamic SQL Pieces Together

Once you can generate SQL from Python data, the full CRUD cycle becomes a sequence of predictable steps. You start with a namedtuple that defines your fields. You use a sample instance to infer SQLite types. You generate a `CREATE TABLE` statement from those pieces. After that, every operation—insert, update, delete, and select—follows the same pattern: assemble SQL from the namedtuple’s metadata and execute it with the instance’s values.

The important part is that nothing is hard‑coded. The table name comes from the namedtuple. The field names come from the namedtuple. The field types come from the sample instance. The values come from the instance itself. If you add a new attribute to the namedtuple, the table definition changes automatically. The insert and update statements change automatically. The entire CRUD cycle adapts to the data model without rewriting any SQL.

This is the general case: SQL becomes a generated artifact, not a handwritten one. Python holds the structure; SQL is assembled from it. The result is a flexible, maintainable pattern that scales cleanly as your data model evolves.


### A General‑Purpose Utility Pattern

Once you understand how dynamic SQL works, the next step is to avoid rewriting the same boilerplate in every project. The goal isn’t to build a full ORM or a complicated abstraction layer. It’s simply to collect the recurring patterns into a small set of helper functions that operate on any namedtuple‑based record.

A useful approach is to treat the namedtuple as the schema and treat each instance as a row. The helpers do the mechanical work: building SQL strings, inferring types, and executing statements. Your application code stays focused on the data itself.

A minimal utility layer usually includes:

- a function to infer SQLite types from Python values
- a function to generate a `CREATE TABLE` statement
- a function to insert a row
- a function to update a row using a dictionary
- a function to delete a row
- a function to select rows

Each helper is small and predictable. None of them contain hard‑coded field names. They all operate on the namedtuple’s metadata. If you change the namedtuple, the helpers adapt automatically.

This pattern gives you the benefits of dynamic SQL without the complexity of a full ORM. It keeps your code maintainable, reduces duplication, and makes the SQL generation logic easy to reason about.


In [ ]:
def sqlite_type(value):
    return {
        "str": "TEXT",
        "int": "INTEGER",
        "float": "REAL",
        "bytes": "BLOB"
    }.get(type(value).__name__, "TEXT")


def create_table(conn, ntuple, sample):
    table = ntuple.__name__.lower() + "s"
    cols = [
        f"{field} {sqlite_type(getattr(sample, field))}"
        for field in ntuple._fields
    ]
    sql = (
        f"CREATE TABLE IF NOT EXISTS {table} ("
        "id INTEGER PRIMARY KEY AUTOINCREMENT, "
        + ", ".join(cols) +
        ");"
    )
    conn.execute(sql)
    conn.commit()
    return table


def insert_row(conn, table, row):
    fields = row._fields
    placeholders = ", ".join(["?"] * len(fields))
    sql = (
        f"INSERT INTO {table} ({', '.join(fields)}) "
        f"VALUES ({placeholders})"
    )
    conn.execute(sql, tuple(row))
    conn.commit()


def update_row(conn, table, name, new_values):
    assignments = ", ".join([f"{k} = ?" for k in new_values])
    params = list(new_values.values()) + [name]
    sql = (
        f"UPDATE {table} SET {assignments} "
        f"WHERE name = ?"
    )
    conn.execute(sql, params)
    conn.commit()


def delete_row(conn, table, name):
    conn.execute(f"DELETE FROM {table} WHERE name = ?", (name,))
    conn.commit()


def select_all(conn, table):
    cur = conn.cursor()
    cur.execute(f"SELECT * FROM {table}")
    return cur.fetchall()


### Exercises: Practicing Dynamic SQL (optional)

These exercises help you internalize the pattern of generating SQL from Python data instead of writing SQL by hand. Each task builds on the same ideas: namedtuple metadata, type inference, and assembling SQL strings from those pieces.

#### 1. Add a New Field to the Namedtuple
Modify the namedtuple by adding a new attribute, such as `mass_kg`.
Create a new sample instance and regenerate the table.
Verify that:
- the new column appears in the table
- `INSERT` and `UPDATE` work without changing any SQL

#### 2. Write a Dynamic `SELECT` with a WHERE Clause
Create a function that selects planets with an orbital period greater than a given number.
Build the SQL string using:
- the table name
- a parameter placeholder
- a comparison operator

Run it for values like `1.0`, `10.0`, and `100.0`.

#### 3. Write a Dynamic `SELECT` That Returns Namedtuples
Fetch rows from the database and convert each row back into a namedtuple instance.
This shows how to round‑trip data between Python and SQLite.

#### 4. Add Ordering to a Dynamic Query
Write a function that selects all planets and orders them by distance from the sun.
Build the `ORDER BY` clause dynamically.

#### 5. Add a New Planet and Update It
Insert a new planet using your dynamic `INSERT`.
Then update two of its fields using your dynamic `UPDATE`.
Verify the changes with a dynamic `SELECT`.

#### 6. Delete a Planet and Confirm the Deletion
Use your dynamic `DELETE` to remove a planet by name.
Run a dynamic `SELECT` to confirm it’s gone.

#### 7. Build a Reusable “Table Manager” Class (Optional)
Wrap your dynamic SQL helpers into a small class that:
- stores the table name
- stores the namedtuple type
- exposes methods like `create`, `insert`, `update`, `delete`, and `select`

This is not an ORM — just a clean way to organize the patterns you’ve already built.

---

These exercises reinforce the core idea:
**SQL is text, and Python is good at generating text.**
Once you treat SQL as data, your code becomes flexible, maintainable, and easy to extend.
